In [1]:
from nba_api.stats.endpoints import leaguedashlineups
import pandas as pd
import time

seasons = ['2025-26'] # Changed to a list
all_lineups = []

for season in seasons: # Iterate over the list
    for group_size in [2, 3, 4, 5]:
        data = leaguedashlineups.LeagueDashLineups(
            season=season,
            measure_type_detailed_defense='Advanced',
            per_mode_detailed='PerGame',
            group_quantity=group_size,
        ).get_data_frames()[0]

        data['season'] = season
        data['group_size'] = group_size
        all_lineups.append(data)
        time.sleep(1)

df = pd.concat(all_lineups)
df.to_csv('nba_lineups.csv', index=False)


In [2]:
df.shape

(8000, 52)

In [3]:
df.columns

Index(['GROUP_SET', 'GROUP_ID', 'GROUP_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING',
       'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT',
       'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT',
       'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE',
       'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK',
       'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK',
       'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK',
       'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK',
       'PACE_RANK', 'PIE_RANK', 'SUM_TIME_PLAYED', 'season', 'group_size'],
      dtype='str')

## Clean the data

In [4]:
from src.get_data import get_player_stats, compute_expected_metric, get_lineups, parse_player_ids, compute_weighted_ts, compute_expected_net

In [5]:
SEASONS       = ['2025-26']          # add more e.g. ['2022-23', '2023-24', '2024-25', '2025-26']
GROUP_SIZES   = [2, 3, 4, 5]
MIN_THRESHOLDS = {2: 100, 3: 75, 4: 50, 5: 30}   # minimum shared minutes to include a lineup

In [6]:
all_lineups = []
all_players = []

for season in SEASONS:
    print(f"\n{'='*50}")
    print(f"Season: {season}")
    print(f"{'='*50}")

    # 1. Individual player stats
    player_df = get_player_stats(season)
    all_players.append(player_df)
    time.sleep(1)

    # 2. Build lookup dicts: player_id → metric
    player_net_lookup = dict(zip(player_df['PLAYER_ID'], player_df['NET_RATING']))
    player_off_lookup = dict(zip(player_df['PLAYER_ID'], player_df['OFF_RATING']))
    player_def_lookup = dict(zip(player_df['PLAYER_ID'], player_df['DEF_RATING']))
    player_pie_lookup = dict(zip(player_df['PLAYER_ID'], player_df['PIE']))
    player_ts_lookup  = dict(zip(player_df['PLAYER_ID'], player_df['TS_PCT']))
    player_usg_lookup = dict(zip(player_df['PLAYER_ID'], player_df['USG_PCT']))

    # 3. Lineup data for each group size
    for group_size in GROUP_SIZES:
        df = get_lineups(season, group_size)

        # Parse player IDs
        df['player_ids'] = df['GROUP_ID'].apply(parse_player_ids)

        # Apply minimum minutes filter
        min_thresh = MIN_THRESHOLDS[group_size]
        df = df[df['MIN'] >= min_thresh].copy()
        print(f"    {len(df)} lineups after {min_thresh}min filter")

        # Compute expected values from individual stats
        df['expected_net_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_net(ids, player_net_lookup)
        )
        df['expected_off_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_off_lookup)
        )
        df['expected_def_rating'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_def_lookup)
        )
        df['expected_pie'] = df['player_ids'].apply(
            lambda ids: compute_expected_metric(ids, player_pie_lookup)
        )
        df['expected_ts_usg_weighted'] = df['player_ids'].apply(
            lambda ids: compute_weighted_ts(ids, player_ts_lookup, player_usg_lookup)
        )

        # Compute synergy deltas
        df['synergy_delta']     = df['NET_RATING'] - df['expected_net_rating']
        df['off_synergy_delta'] = df['OFF_RATING'] - df['expected_off_rating']
        df['def_synergy_delta'] = df['DEF_RATING'] - df['expected_def_rating']   # lower is better
        df['pie_synergy_delta'] = df['PIE']        - df['expected_pie']

        # Convert player_ids list → string for CSV export
        df['player_ids_str'] = df['player_ids'].apply(lambda x: ','.join(x))
        df = df.drop(columns=['player_ids'])

        all_lineups.append(df)


Season: 2025-26
  Fetching individual player stats for 2025-26...
  Fetching 2-man lineups for 2025-26...
    2166 lineups after 100min filter
  Fetching 3-man lineups for 2025-26...
    3256 lineups after 75min filter
  Fetching 4-man lineups for 2025-26...
    1861 lineups after 50min filter
  Fetching 5-man lineups for 2025-26...
    397 lineups after 30min filter


In [16]:
player_df.shape

(546, 15)

In [24]:
df_combined = pd.concat(all_lineups[0:4], ignore_index=True)
df_combined.to_csv('lineups_with_synergy.csv', index=False)

### Create clusters

In [ ]:
from src.cluster_creation import load_data, cluster_players, plot_clusters, build_archetype_lookup, enrich_lineups, archetype_compatibility_matrix, best_lineups_per_team

In [ ]:
players_df, lineups_df = load_data()
players_with_archetypes, scaler, km = cluster_players(players_df)
plot_clusters(players_with_archetypes)

Loading data...
  546 player-season rows
  7680 lineup rows
